# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/M-Raqib-Usman/ML_Internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit

# Load data
csv_files = list(Path(".").rglob("content_refresh_anonymized.csv"))
if not csv_files:
    import subprocess, os
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/M-Raqib-Usman/ML_Internship.git", "repo_temp"], check=True)
    os.chdir("repo_temp")
    csv_files = list(Path(".").rglob("content_refresh_anonymized.csv"))

df = pd.read_csv(csv_files[0])
df["is_declining_label"] = (df["trend_direction"].str.lower() == "down").astype(int)
print("Loaded:", df.shape)

Loaded: (30000, 45)


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

**Actions used:**
- `refresh` — page is stale and still visible; worth editorial review
- `refresh_and_review_ctr` — same as above + low CTR relative to position
- `monitor` — lower priority; keep watching

**Reason codes:**
- `stale_but_visible`
- `low_ctr_visible_page`
- `model_decline_risk`

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Features + model (same as Week 5/6)
feature_cols = [
    "content_age_days", "days_since_last_update", "impressions_90d",
    "avg_position", "ctr", "word_count", "engagement_rate", "sessions_90d"
]
X = df[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
y = df["is_declining_label"].values
groups = df["client_id"].values

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
tr_idx, te_idx = next(gss.split(X, y, groups))

rf = RandomForestClassifier(
    n_estimators=200, max_depth=10, min_samples_leaf=25,
    class_weight="balanced_subsample", random_state=42, n_jobs=-1
)
rf.fit(X.iloc[tr_idx], y[tr_idx])
df["model_score"] = rf.predict_proba(X)[:, 1]

# Rule components
stale   = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
low_ctr = ((df["ctr"] < 1.0) & (df["avg_position"] > 0) & (df["avg_position"] <= 20)).astype(int)

# Reason codes + actions
def assign_reason(row):
    reasons = []
    if row["days_since_last_update"] >= 180 and row["impressions_90d"] >= 500:
        reasons.append("stale_but_visible")
    if row["ctr"] < 1.0 and 0 < row["avg_position"] <= 20 and row["impressions_90d"] >= 500:
        reasons.append("low_ctr_visible_page")
    if row["model_score"] >= 0.65:
        reasons.append("model_decline_risk")
    return ", ".join(reasons) if reasons else "low_priority"

def assign_action(row):
    if "low_ctr_visible_page" in row["reason_code"]:
        return "refresh_and_review_ctr"
    if "stale_but_visible" in row["reason_code"] or "model_decline_risk" in row["reason_code"]:
        return "refresh"
    return "monitor"

df["reason_code"] = df.apply(assign_reason, axis=1)
df["action"] = df.apply(assign_action, axis=1)

# Final ranked queue
queue = df.sort_values("model_score", ascending=False).copy()
queue["rank"] = range(1, len(queue) + 1)

print(queue[["rank", "content_id", "model_score", "action", "reason_code",
             "impressions_90d", "days_since_last_update"]].head(10))

       rank            content_id  model_score                  action  \
14115     1  content_5cda32af644c     0.884242  refresh_and_review_ctr   
17547     2  content_f552433bab3c     0.881208                 refresh   
23015     3  content_1c539e7bac93     0.880245  refresh_and_review_ctr   
29447     4  content_2744dcacbe42     0.875731                 refresh   
28154     5  content_4d76cdb3387b     0.872087  refresh_and_review_ctr   
6559      6  content_4500346533b1     0.870628  refresh_and_review_ctr   
27535     7  content_e04eb9549989     0.870430  refresh_and_review_ctr   
5447      8  content_861205133cbc     0.866014                 refresh   
5732      9  content_b893db8d7619     0.865456  refresh_and_review_ctr   
28121    10  content_ffeed86ef360     0.864252                 refresh   

                                    reason_code  impressions_90d  \
14115  low_ctr_visible_page, model_decline_risk              720   
17547                        model_decline_risk  

## 2. Intended use and limits

**Intended use**  
Content editors / SEO specialists open the ranked list, read the reason codes, and decide which pages to review first when time is limited.  
The output is **decision-support**, not an automatic instruction.

**Limits**
- Built on one anonymized snapshot; patterns may differ across clients and time periods.
- Does not prove that refreshing a page will recover traffic.
- Does not detect algorithm changes, seasonality, or competitor moves.
- Low-volume pages produce noisy signals.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Human review + the no-go list

**What a person must check before acting**
- Is the page still strategically important for the business?
- Was it recently updated outside this data window?
- Is the decline seasonal or caused by a known event?
- Does the suggested action (refresh vs CTR review) match the actual content problem?

**No-go list (never automate)**
- Automatic rewriting or publishing of content
- Automatic deletion / pruning of pages
- Acting on pages with near-zero impressions without human judgment
- Overriding client-specific brand or legal constraints

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Monitoring / retrain triggers

Retrain or re-audit when any of these appear:
- Precision@50 on a fresh client-holdout drops noticeably below the validated range
- Large shift in declining-label base rate
- New content types or clients with very different history depth enter the portfolio
- Feature distributions (impressions, position, freshness) drift significantly
- Editors consistently reject the top of the queue for the same reason

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/.*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
out_dir = Path("work/outputs")
out_dir.mkdir(parents=True, exist_ok=True)

export_cols = [
    "rank", "content_id", "client_id", "model_score", "action", "reason_code",
    "impressions_90d", "days_since_last_update", "avg_position", "ctr",
    "trend_direction", "is_declining_label"
]

queue[export_cols].to_csv(out_dir / "action_playbook_queue.csv", index=False)

# Simple metrics receipt
metrics = {
    "rows_scored": int(len(queue)),
    "refresh_actions": int((queue["action"] == "refresh").sum()),
    "refresh_and_review_ctr": int((queue["action"] == "refresh_and_review_ctr").sum()),
    "monitor_actions": int((queue["action"] == "monitor").sum()),
    "top50_declining_rate": float(queue.head(50)["is_declining_label"].mean())
}
import json
with open(out_dir / "action_playbook_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print("Exported:")
print(" - work/outputs/action_playbook_queue.csv")
print(" - work/outputs/action_playbook_metrics.json")
print(metrics)

Exported:
 - work/outputs/action_playbook_queue.csv
 - work/outputs/action_playbook_metrics.json
{'rows_scored': 30000, 'refresh_actions': 4954, 'refresh_and_review_ctr': 11487, 'monitor_actions': 13559, 'top50_declining_rate': 1.0}


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.